# padding-amount-formula-convT — ex1: predict ConvTranspose2d output size from the padding arg

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `padding-amount-formula-convT`. Running the final beacon cell reports progress against the `CNN: ConvT padding amount formula` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT padding amount formula` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`padding-amount-formula-convT`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "padding-amount-formula-convT"
DD_SUBTOPIC = "CNN: ConvT padding amount formula"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ConvTranspose2d `padding` arg — quick refresher

PyTorch's `nn.ConvTranspose2d(..., padding=P)` does **not** add `P` rows of zero padding to the input. Instead it **removes** `P` rows from the output, equivalent to using effective padding `K - 1 - P` in the underlying flipped-padded conv:

```
effective_pad = K - 1 - P
H_out         = (H_in - 1) * S - 2 * P + K
```

**Why the asymmetric meaning.** ConvT is the adjoint of forward conv. A forward conv with `padding=P` adds `P` to each side of the input — increases output size. Its adjoint subtracts that same `P` from the output of the transposed operation. Same letter, opposite sign at shape-time.

**The gotcha.** Reading `nn.ConvTranspose2d(..., padding=1)` and expecting 'pad input by 1' is the most-confused PyTorch shape bug. `padding=1` actually **shrinks** the output by 2 (1 row per side) relative to `padding=0`.

**The intuition.** Pair a forward `Conv2d(stride=2, K=3, padding=1)` with its inverse `ConvTranspose2d(stride=2, K=3, padding=1, output_padding=1)` — the shapes round-trip exactly. The matching `padding` args 'cancel' as adjoints.

**Quick check (stride 1).** `ConvTranspose2d(IC, OC, K=3, padding=0)` on H_in=4 → H_out = 4 + 2 = 6. Same kernel with `padding=1` → H_out = 4 + 2 - 2 = 4. With `padding=2` → H_out = 4 + 2 - 4 = 2. Each unit of `padding` peels one row off each side of the output.

### Exercise 1 — predict ConvTranspose2d output size from the padding arg

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `H_out = (H_in - 1) * S - 2 * P + K` to predict the spatial output size of `nn.ConvTranspose2d` for given `(H_in, K, S, P)` and verify against the real module on multiple parameter combinations.
> Keywords: ConvTranspose2d, padding, output-shape, adjoint
> ```

**KCs targeted:** `convT-padding-subtracts-from-output`, `convT-output-shape-formula`

Implement `ex1_convT_outlen(h_in, k, s, p)`. Return the spatial output length of a `nn.ConvTranspose2d` with kernel size `k`, stride `s`, padding `p`, and no output_padding, applied to a 1-D input of length `h_in`.

**Formula.**
```
h_out = (h_in - 1) * s - 2 * p + k
```

**Reading the formula.**
- `(h_in - 1) * s + k` is the no-padding case (matches the fractional-stride dilation atom's shape).
- `- 2 * p` is the asymmetric padding subtraction: each unit of `padding=P` PEELS one row off EACH side of the output.
- So `padding` in ConvT2d is the *opposite* of `padding` in Conv2d (which ADDS to the output).

**Mental check.** For `h_in=4, k=3, s=1, p=0` → `(4-1)*1 - 0 + 3 = 6` (output grows from 4 to 6). For the same input + kernel with `p=1` → `(4-1)*1 - 2 + 3 = 4` (output back to 4 — the +2 expansion is exactly cancelled by the -2 padding shrink).

The test exercises many `(h_in, k, s, p)` combinations and compares your prediction against the actual output shape of `nn.ConvTranspose2d` (stride-1 and stride-2 cases).

In [ ]:
def ex1_convT_outlen(h_in: int, k: int, s: int, p: int) -> int:
    """Output length of nn.ConvTranspose2d for given (h_in, k, s, p)."""
    raise NotImplementedError()


def _test_ex1():
    from torch import nn

    # Direct value checks.
    assert ex1_convT_outlen(h_in=4,  k=3, s=1, p=0) == 6,  '(4-1)*1 - 0 + 3 = 6'
    assert ex1_convT_outlen(h_in=4,  k=3, s=1, p=1) == 4,  '(4-1)*1 - 2 + 3 = 4 (padding shrinks)'
    assert ex1_convT_outlen(h_in=4,  k=3, s=1, p=2) == 2,  '(4-1)*1 - 4 + 3 = 2'
    assert ex1_convT_outlen(h_in=5,  k=3, s=2, p=0) == 11, '(5-1)*2 - 0 + 3 = 11'
    assert ex1_convT_outlen(h_in=5,  k=3, s=2, p=1) == 9,  '(5-1)*2 - 2 + 3 = 9'
    assert ex1_convT_outlen(h_in=8,  k=4, s=2, p=1) == 16, '(8-1)*2 - 2 + 4 = 16 (canonical 2x upsample)'
    assert ex1_convT_outlen(h_in=1,  k=5, s=1, p=0) == 5,  'single-pixel input + 5-tap kernel → 5'

    # Sign-direction trap: each +1 padding must DECREASE output by 2,
    # never increase. (This is the most-confused PyTorch shape gotcha.)
    baseline = ex1_convT_outlen(10, 3, 1, p=0)
    for p in [1, 2, 3]:
        assert ex1_convT_outlen(10, 3, 1, p=p) == baseline - 2 * p, (
            f'p={p}: padding must subtract 2*p from output, not add'
        )

    # Cross-check against actual nn.ConvTranspose2d shapes — many combos.
    cases = [
        (4, 3, 1, 0), (4, 3, 1, 1), (5, 3, 2, 1), (8, 4, 2, 1),
        (16, 3, 1, 1), (10, 5, 2, 0), (7, 3, 2, 0), (3, 7, 1, 2),
    ]
    for h_in, k, s, p in cases:
        ct = nn.ConvTranspose2d(in_channels=1, out_channels=1, kernel_size=k, stride=s, padding=p)
        x = t.randn(1, 1, h_in, h_in)
        actual = ct(x).shape[-1]
        predicted = ex1_convT_outlen(h_in, k, s, p)
        assert predicted == actual, (
            f'(h={h_in},k={k},s={s},p={p}): predicted {predicted}, actual {actual}'
        )

    # Conv2d ↔ ConvTranspose2d round-trip:
    # A stride-2 K=3 P=1 Conv2d takes H → ceil(H/2).
    # Pairing with ConvTranspose2d stride=2 K=3 P=1 (output_padding=1) recovers H exactly
    # (when H is even). We verify with the formula version: paired padding cancels.
    h = 16
    # Forward conv2d: (16 + 2*1 - 3) // 2 + 1 = 8 → halves cleanly.
    # ConvT inverse without output_padding: ex1_convT_outlen(8, 3, 2, 1) = (8-1)*2 - 2 + 3 = 15.
    # Off by 1 from h=16 — that's exactly what output_padding=1 corrects (covered elsewhere).
    assert ex1_convT_outlen(8, 3, 2, 1) == 15, 'paired ConvT lands at H-1 — output_padding closes the gap'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_convT_outlen(h_in: int, k: int, s: int, p: int) -> int:
    return (h_in - 1) * s - 2 * p + k
```

**The full PyTorch formula** (including `output_padding=OP` and `dilation=D`):
```
h_out = (h_in - 1) * s - 2 * p + d * (k - 1) + op + 1
```
With `D = 1` and `OP = 0` this reduces to our `(h_in - 1)*s - 2*p + k`. `output_padding` is the extra knob for closing the (stride-2 ⇒ shape rounds down by 1) gap when round-tripping with a forward conv.

**Why `padding` subtracts (the adjoint argument).** ConvTranspose is the adjoint of Conv2d. A forward Conv2d with `padding=P` ADDS `P` zeros on each side of the input → the output grows by `2P`. The adjoint operation must reverse that shape change — so ConvT's `padding=P` SUBTRACTS `2P` from the output. Same param name, opposite shape effect, because they're adjoint.

**Practical recipe.** When pairing a `Conv2d(s, p)` with its inverse `ConvTranspose2d(s, p)`, use **identical** stride and padding values. For stride > 1, also set `output_padding = stride - 1` to recover the exact input shape — otherwise the result is off by 1.

**Why every upsampling network uses `ConvT(K=4, S=2, P=1)`.** Plug in: `(h_in - 1)*2 - 2 + 4 = 2 * h_in` — clean 2× upsample with no off-by-one. This is the GAN / U-Net / diffusion-VAE default.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()